# PySpark - iceberg (Data Lakehouse)

### Ajustando dependências para o PySpark (openjdk-17)

In [1]:
import os
from pathlib import Path
import jdk

ROOT = Path.cwd().parent.parent
JDK_DIR = ROOT / ".jdk"

# só baixa se ainda não existir 🐇
existentes = list(JDK_DIR.glob("jdk-17*")) if JDK_DIR.exists() else []
if existentes:
    java_home = str(existentes[0])
else:
    java_home = jdk.install("17", path=str(JDK_DIR))

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]

### Configurando o PySpark

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("IcebergTest") \
  .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1') \
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
  .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.local.type", "hadoop") \
  .config("spark.sql.catalog.local.warehouse", "spark-warehouse/iceberg") \
  .getOrCreate()

26/10/04 22:08:50 WARN Utils: Your hostname, debian resolves to a loopback address: 127.0.1.1; using 192.168.101.6 instead (on interface wlp62s0)
26/10/04 22:08:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/mateus/.cache/pypoetry/virtualenvs/spark-iceberg-yGLP1Maj-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/mateus/.ivy2/cache
The jars for the packages stored in: /home/mateus/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-372c2317-9fd7-4ba2-a8ac-4c33a012fd34;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 95ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

### Lendo o CSV

In [17]:
CLEAN_CSV = Path.cwd() / "data" / "metacritic_dataset_clean_sep_2025.csv"
RAW_CSV = Path.cwd() / "data" / "metacritic_dataset_raw_sep_2025.csv"

df = (
    spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(RAW_CSV))
)

df.printSchema()
df.show(5)

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- metascore: string (nullable = true)
 |-- user_score: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)

+--------------------+-----------------+------------+---------+----------+--------------+--------------------+-----------------+
|                name|         platform|release_date|metascore|user_score|     developer|           publisher|            genre|
+--------------------+-----------------+------------+---------+----------+--------------+--------------------+-----------------+
|The Legend of Zel...|      Nintendo 64|  1998-11-23|       99|       9.1|      Nintendo|['Nintendo', 'Gra...|Open-World Action|
|         SoulCalibur|        Dreamcast|  1999-09-08|       98|       7.6|         Namco|           ['Namco']|      3D Fighting|
|         SoulCalibur|iOS (iP

### Criando o namespace

In [5]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze")

DataFrame[]

# Validando valores nulos no DataFrame

### Gerando um DataFrame com as linhas que possuem valores nulos

In [18]:
from functools import reduce
from pyspark.sql import functions as F

null_condition = reduce(lambda x, y: x | y, [F.col(c).isNull() for c in df.columns]) # gera algo como (col1.isNull() OR col2.isNull() OR col3.isNull() etc...)

null_df = df.filter(null_condition).cache()
null_df.count()

null_df.show()

+--------------------+----------------+------------+---------+----------+--------------------+--------------------+-----------------+
|                name|        platform|release_date|metascore|user_score|           developer|           publisher|            genre|
+--------------------+----------------+------------+---------+----------+--------------------+--------------------+-----------------+
|  Super Mario Galaxy| Nintendo Switch|  2007-11-12|      tbd|      NULL|            Nintendo|        ['Nintendo']|    3D Platformer|
|Super Mario Galaxy 2| Nintendo Switch|  2010-05-23|      tbd|      NULL|  Nintendo EAD Tokyo|['Nintendo', 'iQue']|    3D Platformer|
|Tony Hawk's Pro S...|     Nintendo 64|  2001-10-30|     NULL|      NULL|Neversoft Enterta...|['Activision', 'S...|          Skating|
|       Metroid Prime|             Wii|  2002-11-17|     NULL|      NULL|       Retro Studios|        ['Nintendo']|              FPS|
|Grand Theft Auto III|            Xbox|  2001-10-22|     NULL|

### Gravando os valores nulos em uma tabela alternativa

In [14]:
null_df.writeTo("local.bronze.quarentena").createOrReplace()

26/10/04 20:21:15 WARN HadoopTableOperations: Error reading version hint file spark-warehouse/iceberg/bronze/quarentena/metadata/version-hint.text
java.io.FileNotFoundException: File spark-warehouse/iceberg/bronze/quarentena/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:320)
	at org.apache.iceberg.hado

### Gerando um DataFrames sem valores nulos

In [15]:
valid_df = df.filter(~null_condition)
valid_df.show(5)

+--------------------+-----------------+------------+---------+----------+--------------+--------------------+-----------------+
|                name|         platform|release_date|metascore|user_score|     developer|           publisher|            genre|
+--------------------+-----------------+------------+---------+----------+--------------+--------------------+-----------------+
|The Legend of Zel...|      Nintendo 64|  1998-11-23|       99|       9.1|      Nintendo|['Nintendo', 'Gra...|Open-World Action|
|         SoulCalibur|        Dreamcast|  1999-09-08|       98|       7.6|         Namco|           ['Namco']|      3D Fighting|
|         SoulCalibur|iOS (iPhone/iPad)|  1999-09-08|       73|       7.8|         Namco|           ['Namco']|      3D Fighting|
|         SoulCalibur|         Xbox 360|  1999-09-08|       79|       7.3|         Namco|           ['Namco']|      3D Fighting|
| Grand Theft Auto IV|    PlayStation 3|  2008-04-29|       98|       8.0|Rockstar North|['Rockst

### Criando a tabela principal

In [25]:
from pyspark.sql.types import (StructType, StructField, StringType, DateType)

schema = StructType([
    StructField("name", StringType(), False),
    StructField("platform", StringType(), False),
    StructField("release_date", DateType(), False),
    StructField("metascore", StringType(), False),
    StructField("user_score", StringType(), False),
    StructField("developer", StringType(), False),
    StructField("publisher", StringType(), False),
    StructField("genre", StringType(), False)
])

spark.createDataFrame([], schema).writeTo("local.bronze.jogos").createOrReplace()

valid_df.writeTo("local.bronze.jogos").option("check-nullability", "false").append()

26/10/04 21:06:39 WARN HadoopTableOperations: Error reading version hint file spark-warehouse/iceberg/bronze/jogos/metadata/version-hint.text
java.io.FileNotFoundException: File spark-warehouse/iceberg/bronze/jogos/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:320)
	at org.apache.iceberg.hadoop.HadoopT

### Testando se tudo deu certo

In [26]:
spark.sql("SHOW TABLES IN local.bronze").show()
spark.table("local.bronze.jogos").show(5)
spark.sql("SELECT * FROM local.bronze.jogos.snapshots").show()

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|   bronze|quarentena|      false|
|   bronze|     jogos|      false|
+---------+----------+-----------+

+--------------------+-----------------+------------+---------+----------+--------------+--------------------+-----------------+
|                name|         platform|release_date|metascore|user_score|     developer|           publisher|            genre|
+--------------------+-----------------+------------+---------+----------+--------------+--------------------+-----------------+
|The Legend of Zel...|      Nintendo 64|  1998-11-23|       99|       9.1|      Nintendo|['Nintendo', 'Gra...|Open-World Action|
|         SoulCalibur|        Dreamcast|  1999-09-08|       98|       7.6|         Namco|           ['Namco']|      3D Fighting|
|         SoulCalibur|iOS (iPhone/iPad)|  1999-09-08|       73|       7.8|         Namco|           ['Namco']|      3D Fighting|
|         SoulC

# Operações de CRUD padrão com PySpark

## Adicionando jogos de 2026

In [10]:
from datetime import date

new_games_df = spark.createDataFrame([
    ("Gears of War: E-Day", "Xbox Series X", date(2026, 10, 6), "87", "8.5", "The Coalition", "[Xbox Game Studio]", "Third Person Shooter"),
    ("Gears of War: E-Day", "PC", date(2026, 10, 6), "83", "tbd", "The Coalition", "[Xbox Game Studio]", "Third Person Shooter"),
    ("The Witcher 3: Wild Hunt - Remastered", "PC", date(2026, 9, 29), "93", "8.5", "CD Projekt Red Studio", "[CD Projekt, Red, Studio]", "Action RPG"),
    ("The Witcher 3: Wild Hunt - Remastered", "Nintendo Switch 2", date(2026, 9, 29), "95", "7.7", "CD Projekt Red Studio", "[CD Projekt, Red, Studio]", "Action RPG")
],
["name", "platform", "release_date", "metascore", "user_score", "developer", "publisher", "genre"])

new_games_df.writeTo("local.bronze.jogos").option("check-nullability", "false").append()

### Alterando jogos nulos com sql

In [12]:
spark.sql("UPDATE local.bronze.quarentena SET user_score = 7.9 WHERE name = 'Super Mario Galaxy' AND platform = 'Nintendo Switch'")

DataFrame[]

### Alterando jogos com um DataFrame

In [ ]:
pivot_df = spark.table("local.bronze.quarentena")

another_new_df = (
    pivot_df
    .withColumn("user_score", F.when((F.col("name") == "Super Mario Galaxy 2") & (F.col("platform") == "Nintendo Switch"), "8.4")
    .otherwise(F.col("user_score")))
    .withColumn("metascore", F.when((F.col("name") == "Tony Hawk's Pro Skater 3") & (F.col("platform") == "Nintendo 64"), "tbd")
    .otherwise(F.col("metascore")))
    .withColumn("user_score", F.when((F.col("name") == "Tony Hawk's Pro Skater 3") & (F.col("platform") == "Nintendo 64"), "tbd")
    .otherwise(F.col("user_score")))
    .withColumn("user_score", F.when((F.col("name") == "Metroid Prime") & (F.col("platform") == "Wii"), "7.8")
    .otherwise(F.col("user_score")))
    .withColumn("metascore", F.when((F.col("name") == "Metroid Prime") & (F.col("platform") == "Wii"), "tbd")
    .otherwise(F.col("metascore")))
).filter(~null_condition)

another_new_df.writeTo("local.bronze.jogos").option("check-nullability", "false").append()

another_new_df.show()

del pivot_df # Limpa o DataFrame da memória

+--------------------+---------------+------------+---------+----------+--------------------+--------------------+-------------+
|                name|       platform|release_date|metascore|user_score|           developer|           publisher|        genre|
+--------------------+---------------+------------+---------+----------+--------------------+--------------------+-------------+
|  Super Mario Galaxy|Nintendo Switch|  2007-11-12|      tbd|       7.9|            Nintendo|        ['Nintendo']|3D Platformer|
|Super Mario Galaxy 2|Nintendo Switch|  2010-05-23|      tbd|       8.4|  Nintendo EAD Tokyo|['Nintendo', 'iQue']|3D Platformer|
|Tony Hawk's Pro S...|    Nintendo 64|  2001-10-30|      tbd|       tbd|Neversoft Enterta...|['Activision', 'S...|      Skating|
|       Metroid Prime|            Wii|  2002-11-17|      tbd|       7.8|       Retro Studios|        ['Nintendo']|          FPS|
+--------------------+---------------+------------+---------+----------+--------------------+----

### Deletando jogos com MERGE INTO

In [26]:
another_new_df.createOrReplaceTempView("updated")

spark.sql("MERGE INTO local.bronze.quarentena q USING updated u ON q.name = u.name AND q.platform = u.platform WHEN MATCHED THEN DELETE")

spark.table("local.bronze.quarentena").show()

+--------------------+----------------+------------+---------+----------+--------------------+--------------------+-----------------+
|                name|        platform|release_date|metascore|user_score|           developer|           publisher|            genre|
+--------------------+----------------+------------+---------+----------+--------------------+--------------------+-----------------+
|Grand Theft Auto III|            Xbox|  2001-10-22|     NULL|      NULL|          DMA Design|['Rockstar Games'...|Open-World Action|
|Grand Theft Auto III|   PlayStation 4|  2001-10-22|     NULL|      NULL|          DMA Design|['Rockstar Games'...|Open-World Action|
|Grand Theft Auto III|Game Boy Advance|  2001-10-22|     NULL|      NULL|          DMA Design|['Rockstar Games'...|Open-World Action|
|       GoldenEye 007|        Xbox One|  1997-08-25|     NULL|      NULL|           Rare Ltd.|['Nintendo', 'Gra...|              FPS|
|       GoldenEye 007|   Xbox Series X|  1997-08-25|     NULL|